In [ ]:
# 展示label种类和数量
# 展示年份差异
# 展示back差异

In [ ]:
import pandas as pd
import numpy as np
import random
import torch
from tqdm import tqdm
from torch import optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from umap import UMAP
import plotly.express as px
from deepview.calculate_results.data.umineko.umineko_data import (
    read_umineko_path,
    extract_data_from_year_back,
    label_dict,
)
import os
from deepview.calculate_results.models.utils import (
    sliding_window,
    data_loader_umineko,
    MSEloss_weighted,
    MSEloss,
    # torch,
    AE_eval_time_series,
    AE_train_time_series_resnet,
    # np,
    # tqdm
    Autoencoder3d,
    plot_reconstruction_result,
majority_value,
)
def set_random_seed(seed):
    # Set seed for Python's random module
    random.seed(seed)

    # Set seed for NumPy
    np.random.seed(seed)

    # Set seed for PyTorch
    torch.manual_seed(seed)

    # If using CUDA, set seed for GPU as well
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU setups

# Set a fixed random seed
seed_value = 2025
set_random_seed(seed_value)

def gaussian_std(X):
    mean_val = np.mean(X.astype(float), axis=0)
    std_val = np.std(X.astype(float), axis=0)
    X_standardized = (X - mean_val) / np.maximum(std_val, 10 ** -5)
    return X_standardized

In [ ]:
# Data['all_data'] = train_x
# Data['all_label'] = train_y
# if not os.path.exists(current_path + '/Datasets/Skoda/'):
#     os.makedirs(current_path + '/Datasets/Skoda/')
# np.save(current_path + '/Datasets/Skoda/Skoda.npy', Data, allow_pickle=True)

In [ ]:
current_path = os.getcwd()
data_path = current_path + '/Datasets/Skoda/Skoda.npy'
Data = np.load(data_path, allow_pickle=True)
train_x = Data['all_data']
train_y = Data['all_label'] 

In [ ]:
train_x.shape

In [ ]:
train_y.shape

In [ ]:
len_sw = 50
tmp_b = sliding_window(np.concatenate([train_x, train_y.reshape(-1,1)],axis=1), len_sw, len_sw)
data_b = np.transpose(tmp_b[:, :, :-1], (0, 2, 1))  # [B, Len, dim-1] -> [B, dim-1, Len]
label_b = tmp_b[:, :, -1]  # [B, Len]

batch_size = 1024
device = 'cuda:1'
train_set_r = data_loader_umineko(data_b.astype(float), label_b.astype(int), device=device)
train_loader = DataLoader(train_set_r, batch_size=batch_size,
                          shuffle=False, drop_last=False)


In [ ]:
model = Autoencoder3d()
model = model.to(device)
criterion = MSEloss()

criterion = criterion.to(device)

learning_rate = 0.001
# training
start_epoch = 0
num_epochs = 1000

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
# learning rate update per epoch

training_loss = []
for epoch in tqdm(range(start_epoch, num_epochs)):

    losses = AE_train_time_series_resnet(train_loader, model, criterion, optimizer, epoch, scheduler, device)
    training_loss.append(np.average(losses))
    if (epoch % 100 == 0) or (epoch == num_epochs - 1):
        # Print the learning rate
        for param_group in optimizer.param_groups:
            print("Learning Rate:", param_group['lr'])
        #     print('loss of the ' + str(epoch) + '-th training epoch is :' + losses.__str__())
        # reconstruction result
        representation_list, sample_list, pred_list, label_list = \
            AE_eval_time_series(train_loader, model, device)
        plot_reconstruction_result(representation_list, sample_list, pred_list, label_list, 'train_epoch_%s' % str(epoch))